# FPL Transcript Data Generator

Set `gameweek` to the upcoming gameweek, add YouTube transcripts to `gw/gw{gameweek}/transcripts/`, then run this notebook top to bottom before generating a report.

It segments each transcript to retain only FPL analysis, resolves ambiguous player names, then extracts player recommendations, captaincy mentions, and qualitative themes via an LLM. It then resolves player names to FPL ids, and aggregates sentiment/captaincy scores.

The output is a report-ready file at:

```text
gw/gw3/data/gw_3_transcript_analysis.json
```

which `fpl_report.py` reads alongside the aggregated player form data. Raw LLM responses are cached per pass in `gw/gw3/transcript_cache/segmentation_cache/`, `gw/gw3/transcript_cache/name_resolution_cache/`, and `gw/gw3/transcript_cache/extraction_cache/` so re-running this notebook is cheap.

In [1]:
import os

# Ensure all relative paths (gw/, lookup/, reports/) resolve from the project root,
# regardless of whether the notebook is opened from notebooks/ or notebooks/transcripts_processor/.
cwd = os.getcwd()
if os.path.basename(cwd) == "transcripts_processor":
    os.chdir("../..")

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/augustustranter/Documents/Programming/Projects/fpl/fpl_agent


In [2]:
# This variable indicates the gameweek for which we are generating a report.
# E.g. if gameweek = 3, transcripts in gw/gw3/transcripts/ are analysed.
gameweek = 4

In [3]:
import hashlib
import json
import os
import re
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from difflib import get_close_matches
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

data_dir = PROJECT_ROOT / "gw" / f"gw{gameweek}" / "data"
transcript_dir = PROJECT_ROOT / "gw" / f"gw{gameweek}" / "transcripts"
transcript_cache_dir = PROJECT_ROOT / "gw" / f"gw{gameweek}" / "transcript_cache"
segmentation_cache_dir = transcript_cache_dir / "segmentation_cache"
name_resolution_cache_dir = transcript_cache_dir / "name_resolution_cache"
extraction_cache_dir = transcript_cache_dir / "extraction_cache"
aggregate_path = data_dir / f"gw_{gameweek}_aggregated_player_form.json"
element_lookup_path = PROJECT_ROOT / "lookup" / "element_lookup.json"
output_path = data_dir / f"gw_{gameweek}_transcript_analysis.json"
extractions_output_path = data_dir / f"gw_{gameweek}_transcript_extractions.json"
if not aggregate_path.exists():
    raise FileNotFoundError(
        f"Missing {aggregate_path.relative_to(PROJECT_ROOT)}. Run data_generator.ipynb for GW{gameweek} first."
    )
if not element_lookup_path.exists():
    raise FileNotFoundError(f"Missing {element_lookup_path.relative_to(PROJECT_ROOT)}")

with aggregate_path.open(encoding="utf-8") as file:
    players = json.load(file)
with element_lookup_path.open(encoding="utf-8") as file:
    element_lookup = json.load(file)

player_names = {player["name"]: int(player_id) for player_id, player in players.items()}
canonical_player_names = {player["name"]: int(player_id) for player_id, player in element_lookup.items()}
canonical_player_id_to_name = {player_id: name for name, player_id in canonical_player_names.items()}

print(f"Loaded {len(player_names)} players from {aggregate_path.relative_to(PROJECT_ROOT)}")
print(
    f"Loaded {len(canonical_player_names)} canonical player names from "
    f"{element_lookup_path.relative_to(PROJECT_ROOT)}"
 )
print(f"Segmentation cache: {segmentation_cache_dir.relative_to(PROJECT_ROOT)}")
print(f"Name resolution cache: {name_resolution_cache_dir.relative_to(PROJECT_ROOT)}")
print(f"Extraction cache: {extraction_cache_dir.relative_to(PROJECT_ROOT)}")
print(f"Transcript extractions output: {extractions_output_path.relative_to(PROJECT_ROOT)}")


Loaded 656 players from gw/gw4/data/gw_4_aggregated_player_form.json
Loaded 656 canonical player names from lookup/element_lookup.json
Segmentation cache: gw/gw4/transcript_cache/segmentation_cache
Name resolution cache: gw/gw4/transcript_cache/name_resolution_cache
Extraction cache: gw/gw4/transcript_cache/extraction_cache
Transcript extractions output: gw/gw4/data/gw_4_transcript_extractions.json


# 1. SEGMENTATION

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import importlib

transcript_segmentation = importlib.import_module("notebooks.transcripts_processor.1_segmentation")
transcript_segmentation = importlib.reload(transcript_segmentation)
build_analysis_text = transcript_segmentation.build_analysis_text
collect_transcripts = transcript_segmentation.collect_transcripts
segment_transcript = transcript_segmentation.segment_transcript
MAX_CONCURRENT_TRANSCRIPTS = transcript_segmentation.MAX_CONCURRENT_TRANSCRIPTS
SEGMENTATION_TIMEOUT_SECONDS = transcript_segmentation.SEGMENTATION_TIMEOUT_SECONDS
SEGMENTATION_MAX_RETRIES = transcript_segmentation.SEGMENTATION_MAX_RETRIES

model = "gpt-4.1-mini"

load_dotenv(PROJECT_ROOT / ".env")
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to .env.")

client = OpenAI(
    api_key=api_key,
    timeout=SEGMENTATION_TIMEOUT_SECONDS,
    max_retries=SEGMENTATION_MAX_RETRIES,
 )

transcripts = collect_transcripts(transcript_dir) if transcript_dir.exists() else []
sources = [transcript["source"] for transcript in transcripts]
print(f"Found {len(transcripts)} transcript(s) in {transcript_dir.relative_to(PROJECT_ROOT)}")

segment_records: list[dict] = []
failed_segment_sources: list[dict[str, str]] = []
if transcripts:
    max_workers = min(MAX_CONCURRENT_TRANSCRIPTS, len(transcripts))
    print(
        f"Segmenting with {max_workers} worker(s), a {SEGMENTATION_TIMEOUT_SECONDS}s request timeout, "
        f"and {SEGMENTATION_MAX_RETRIES} automatic retries"
    )
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(segment_transcript, client, model, segmentation_cache_dir, transcript["text"]): transcript
            for transcript in transcripts
        }
        for completed, future in enumerate(as_completed(futures), start=1):
            transcript = futures[future]
            source = transcript["source"]
            try:
                segments = future.result()
            except Exception as exc:
                failed_segment_sources.append(
                    {"source": source, "error": f"{type(exc).__name__}: {exc}"}
                )
                print(f"[{completed}/{len(transcripts)}] FAILED {source}: {type(exc).__name__}: {exc}")
                continue

            analysis_text = build_analysis_text(segments)
            segment_records.append(
                {
                    "source": source,
                    "segments": segments,
                    "analysis_text": analysis_text,
                }
            )
            retained_count = sum(segment.get("type") == "FPL_ANALYSIS" for segment in segments)
            print(
                f"[{completed}/{len(transcripts)}] Completed {source}: "
                f"{len(segments)} section(s), {retained_count} retained"
            )
    segment_records.sort(key=lambda record: sources.index(record["source"]))

successful_sources = [record["source"] for record in segment_records]
segment_output = {
    "sources": successful_sources,
    "segment_records": segment_records,
    "failed_sources": failed_segment_sources,
}
print(
    f"Prepared {len(segment_records)} segmented transcript(s) in memory; "
    f"{len(failed_segment_sources)} failed"
 )
if failed_segment_sources:
    print("Failed sources can be retried by running this cell again; completed sources will use the cache:")
    for failure in failed_segment_sources:
        print(f"- {failure['source']}: {failure['error']}")
print(f"Segmentation cache used: {segmentation_cache_dir.relative_to(PROJECT_ROOT)}")

Found 21 transcript(s) in gw/gw4/transcripts
Prepared 21 segmented transcript(s) in memory
Segmentation cache used: gw/gw4/transcript_cache/segmentation_cache


In [11]:
from dotenv import load_dotenv
from openai import OpenAI
import importlib
from concurrent.futures import ThreadPoolExecutor, as_completed

transcript_name_resolution = importlib.import_module("notebooks.transcripts_processor.2_name_resolution")
transcript_name_resolution = importlib.reload(transcript_name_resolution)
build_name_resolution_helpers = transcript_name_resolution.build_name_resolution_helpers
load_name_aliases = transcript_name_resolution.load_name_aliases
MAX_CONCURRENT_NAME_RESOLUTION = transcript_name_resolution.MAX_CONCURRENT_NAME_RESOLUTION

model = "gpt-4.1-mini"

NAME_ALIASES = load_name_aliases(PROJECT_ROOT)
PLAYER_METADATA = {
    int(player_id): {
        "team": str(player.get("team", "")),
        "position": int(player.get("position", 0) or 0),
    }
    for player_id, player in players.items()
}
name_resolution = build_name_resolution_helpers(
    name_aliases=NAME_ALIASES,
    canonical_player_names=canonical_player_names,
    canonical_player_id_to_name=canonical_player_id_to_name,
    player_metadata=PLAYER_METADATA,
 )
resolve_transcript_names = name_resolution.resolve_transcript_names

segment_records = globals().get("segment_records")
if segment_records is None:
    raise RuntimeError("Run the segment pass first.")

print(f"Loaded {len(segment_records)} segmented transcript(s) from memory")

load_dotenv(PROJECT_ROOT / ".env")
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to .env.")

client = OpenAI(api_key=api_key)

name_corrections_by_source: dict[str, list[dict]] = {}
name_corrections: list[dict] = []
resolution_inputs = [
    record
    for record in segment_records
    if str(record.get("analysis_text", "")).strip()
]
if resolution_inputs:
    max_workers = min(MAX_CONCURRENT_NAME_RESOLUTION, len(resolution_inputs))
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(
                resolve_transcript_names,
                client,
                model,
                name_resolution_cache_dir,
                str(record.get("source", "")),
                str(record.get("analysis_text", "")).strip(),
            ): record
            for record in resolution_inputs
        }
        for future in as_completed(futures):
            record = futures[future]
            source = str(record.get("source", ""))
            source_name_corrections = future.result()
            name_corrections_by_source[source] = source_name_corrections
            name_corrections.extend(source_name_corrections)

name_corrections_output_path = data_dir / f"gw_{gameweek}_name_corrections.json"
name_corrections_output = {
    "sources": segment_output.get("sources", []),
    "count": len(name_corrections),
    "resolved_count": sum(1 for item in name_corrections if item["resolution_status"] == "resolved"),
    "name_corrections": name_corrections,
    "name_corrections_by_source": name_corrections_by_source,
}
name_corrections_output_path.write_text(
    json.dumps(name_corrections_output, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Name corrections written to {name_corrections_output_path.relative_to(PROJECT_ROOT)}")
print(f"Name resolution cache used: {name_resolution_cache_dir.relative_to(PROJECT_ROOT)}")


Loaded 4 segmented transcript(s) from memory
Name corrections written to gw/gw3/data/gw_3_name_corrections.json
Name resolution cache used: gw/gw3/transcript_cache/name_resolution_cache


In [12]:
from dotenv import load_dotenv
from openai import OpenAI
import importlib

transcript_name_resolution = importlib.import_module("notebooks.transcripts_processor.2_name_resolution")
transcript_name_resolution = importlib.reload(transcript_name_resolution)
build_name_resolution_helpers = transcript_name_resolution.build_name_resolution_helpers
load_name_aliases = transcript_name_resolution.load_name_aliases
clamp = transcript_name_resolution.clamp

transcript_extraction = importlib.import_module("notebooks.transcripts_processor.3_extraction")
transcript_extraction = importlib.reload(transcript_extraction)
extract_from_transcript = transcript_extraction.extract_from_transcript
write_transcript_extractions_output = transcript_extraction.write_transcript_extractions_output

model = "gpt-4.1-mini"
MIN_MENTIONS_FOR_FULL_CONFIDENCE = 3

NAME_ALIASES = load_name_aliases(PROJECT_ROOT)
PLAYER_METADATA = {
    int(player_id): {
        "team": str(player.get("team", "")),
        "position": int(player.get("position", 0) or 0),
    }
    for player_id, player in players.items()
}
name_resolution = build_name_resolution_helpers(
    name_aliases=NAME_ALIASES,
    canonical_player_names=canonical_player_names,
    canonical_player_id_to_name=canonical_player_id_to_name,
    player_metadata=PLAYER_METADATA,
 )
resolve_supported_player_id = name_resolution.resolve_supported_player_id
PLAYER_ID_TO_NAME = {player_id: name for name, player_id in player_names.items()}

segment_records = globals().get("segment_records")
if segment_records is None:
    raise RuntimeError("Run the segment pass first.")

name_corrections_by_source = globals().get("name_corrections_by_source")
if name_corrections_by_source is None:
    raise RuntimeError("Run the name pass first.")

load_dotenv(PROJECT_ROOT / ".env")
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to .env.")

client = OpenAI(api_key=api_key)

recommendations: list[dict] = []
captaincy_mentions: list[dict] = []
themes: list[tuple[str, str]] = []

for record in segment_records:
    analysis_text = str(record.get("analysis_text", "")).strip()
    if not analysis_text:
        continue
    source = str(record.get("source", ""))
    source_name_corrections = name_corrections_by_source.get(source, [])
    parsed = extract_from_transcript(client, model, extraction_cache_dir, source, analysis_text, source_name_corrections)

    for recommendation in parsed.get("recommendations", []):
        recommendation["source"] = source
        recommendations.append(recommendation)
    for mention in parsed.get("captaincy", []):
        mention["source"] = source
        captaincy_mentions.append(mention)
    for theme in parsed.get("themes", []):
        text = str(theme).strip()
        if text:
            themes.append((text, source))

transcript_extractions_output = write_transcript_extractions_output(
    extractions_output_path,
    sources=segment_output.get("sources", []),
    recommendations=recommendations,
    captaincy=captaincy_mentions,
    themes=themes,
)

print(
    f"Extracted {len(recommendations)} recommendations, {len(captaincy_mentions)} captaincy mentions, "
    f"and {len(themes)} themes"
 )
print(
    f"Transcript extractions written to {extractions_output_path.relative_to(PROJECT_ROOT)} "
    f"({transcript_extractions_output['counts']['sources']} source(s))"
 )
print(f"Extraction cache used: {extraction_cache_dir.relative_to(PROJECT_ROOT)}")

sentiment: dict[int, dict] = {}
stance_values = {
    "buy": 1.0,
    "consider": 0.6,
    "hold": 0.2,
    "avoid": -0.8,
    "sell": -1.0,
}
for recommendation in recommendations:
    player_id = resolve_supported_player_id(recommendation, player_names)
    if player_id is None:
        continue
    stance = str(recommendation.get("stance", "")).lower()
    if stance not in stance_values:
        continue
    confidence = clamp(float(recommendation.get("confidence", 1)), 1, 5)
    entry = sentiment.setdefault(
        player_id,
        {
            "player_name": PLAYER_ID_TO_NAME[player_id],
            "weighted_stances": [],
            "evidence": [],
        },
    )
    entry["weighted_stances"].append(stance_values[stance] * confidence / 5)
    entry["evidence"].append({
        "stance": stance,
        "sentiment": str(recommendation.get("sentiment", "")),
        "recommendation_strength": recommendation.get("recommendation_strength"),
        "rationale": str(recommendation.get("rationale", "")),
        "supporting_quote": str(recommendation.get("supporting_quote", "")),
        "reasons": recommendation.get("reasons", []),
        "risks": recommendation.get("risks", []),
        "time_horizon": str(recommendation.get("time_horizon", "unspecified")),
        "source": str(recommendation.get("source", "")),
        "timestamp": recommendation.get("timestamp"),
    })

for entry in sentiment.values():
    weighted_stances = entry["weighted_stances"]
    mention_count = len(weighted_stances)
    average_stance = sum(weighted_stances) / mention_count
    decay = min(1.0, mention_count / MIN_MENTIONS_FOR_FULL_CONFIDENCE)
    entry["score"] = round(clamp(50 + average_stance * 50 * decay), 1)
    entry["mention_count"] = mention_count
    supportive = sum(1 for value in weighted_stances if value > 0)
    bearish = sum(1 for value in weighted_stances if value < 0)
    entry["disagreement"] = (
        f"{supportive} supportive vs {bearish} bearish" if supportive and bearish else None
    )
    del entry["weighted_stances"]

captaincy_weights = {
    "triple_captain": 1.0,
    "captain": 0.8,
    "vice_captain": 0.4,
    "avoid_captain": -1.0,
}


Extracted 41 recommendations, 14 captaincy mentions, and 30 themes
Transcript extractions written to gw/gw3/data/gw_3_transcript_extractions.json (4 source(s))
Extraction cache used: gw/gw3/transcript_cache/extraction_cache
